# SS7 content regex match report

Scans every raw SS7 CDR file, keeps `message_type in [2, 3]` (MT_request /
MO — the two message types that actually carry content, see
`ingestion/ss7.py`'s module docstring), and searches decoded content for a
regex pattern.

**Column mapping** (raw SS7 CSV → task spec names):

| task spec   | real column       |
|-------------|--------------------|
| messageType | `message_type`     |
| dcs         | `dcs`               |
| callingGT   | `calling_gt`        |
| calledGT    | `called_gt`         |
| smsPayload  | `decoded_content`   |
| timestamp   | `time_stamp`        |

**Why `decoded_content` directly, no hex→ASCII step here**: `ingestion/ss7.py`
(`_decode_row`'s docstring) already verified `decoded_content` matches
`ingestion/dcs_codecs.py::decode_by_dcs()`'s own decode of the raw hex
`content`/`raw_user_data` field exactly, on a real multipart sample — unlike
SMPP, where `decoded_content` was found to leak UDH framing bytes and is
NOT trusted. So for SS7, reusing the upstream `decoded_content` column is
equivalent to decoding the hex payload ourselves, without re-implementing
UDH-stripping + DCS codec selection here. Coverage caveat: rows where
`decoded_content` is null but `content`/`raw_user_data` is not (genuinely
undecoded rows) are reported separately below, not silently dropped.

Processes files one at a time (48 files, ~4GB raw) with only the needed
columns loaded, instead of concatenating the full corpus into memory.

In [1]:
import re
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_SS7_DIR = PROJECT_ROOT / "data" / "raw" / "SS7"

# Anchored to the END of decoded_content ($) - only rows where the
# "(12 alphanumeric chars)" token is the LAST thing in the message, with
# nothing trailing after the closing paren. Leading bytes/text before the
# token are fine (real samples: "\xd2(T2ZKaMjKhf4w)"). This excludes rows
# where the same 12-char shape coincidentally wraps a real dictionary/brand
# word embedded in an actual sentence (e.g. "...SUBRAMONIAN(CREDITXPRESS)
# \n\nSila jelaskan bayaran..." - a real Malay debt-collection-threat SMS,
# not noise) - those have trailing content after the parens and are real
# message text, not the opaque-token artifact this pattern targets.
PATTERN = re.compile(r"\([A-Za-z0-9]{12}\)$")
MESSAGE_TYPES_TO_CHECK = [2, 3]  # MT_request, MO - see markdown above

USECOLS = [
    "time_stamp", "message_type", "dcs", "calling_gt", "called_gt",
    "decoded_content", "content", "raw_user_data", "reference",
]

raw_files = sorted(RAW_SS7_DIR.glob("*/*.csv"))
print(f"{len(raw_files)} raw SS7 files found under {RAW_SS7_DIR}")

48 raw SS7 files found under C:\Users\IshitaGodani\Documents\projects\spam-detection-prototype\data\raw\SS7


In [2]:
match_chunks = []
total_rows_seen = 0
total_filtered_rows = 0  # message_type in [2, 3]
total_undecoded_filtered_rows = 0  # filtered rows with no decoded_content
# but real content/raw_user_data present - NOT searched, reported as a gap

for i, path in enumerate(raw_files, start=1):
    df = pd.read_csv(path, usecols=lambda c: c in USECOLS, low_memory=False)
    total_rows_seen += len(df)

    filtered = df[df["message_type"].isin(MESSAGE_TYPES_TO_CHECK)]
    total_filtered_rows += len(filtered)

    has_payload = filtered["content"].notna() | filtered["raw_user_data"].notna()
    undecoded = filtered["decoded_content"].isna() & has_payload
    total_undecoded_filtered_rows += int(undecoded.sum())

    hit = filtered["decoded_content"].str.contains(PATTERN, regex=True, na=False)
    if hit.any():
        hit_rows = filtered.loc[hit, [
            "time_stamp", "message_type", "dcs", "calling_gt", "called_gt",
            "decoded_content", "reference",
        ]].copy()
        hit_rows["matched_token"] = hit_rows["decoded_content"].str.extract(r"(\([A-Za-z0-9]{12}\))$")
        match_chunks.append(hit_rows)

    if i % 10 == 0 or i == len(raw_files):
        print(f"[{i}/{len(raw_files)}] {path.name}: {len(df)} rows, "
              f"{len(filtered)} in scope, {len(match_chunks)} files with matches so far")

matches = (
    pd.concat(match_chunks, ignore_index=True)
    if match_chunks else
    pd.DataFrame(columns=["time_stamp", "message_type", "dcs", "calling_gt", "called_gt", "decoded_content", "reference", "matched_token"])
)

print(f"\ntotal rows scanned: {total_rows_seen:,}")
print(f"rows in scope (message_type in {MESSAGE_TYPES_TO_CHECK}): {total_filtered_rows:,}")
print(f"in-scope rows with a real payload but no decoded_content (not searched): {total_undecoded_filtered_rows:,}")
print(f"total pattern matches (row-level, token anchored to end of message): {len(matches):,}")

[10/48] stg_ss7_20260802_0900.csv: 266177 rows, 57851 in scope, 10 files with matches so far


[20/48] stg_ss7_20260802_1900.csv: 339103 rows, 76669 in scope, 20 files with matches so far


[30/48] stg_ss7_20260803_0500.csv: 157936 rows, 28456 in scope, 30 files with matches so far


[40/48] stg_ss7_20260803_1500.csv: 542639 rows, 157581 in scope, 40 files with matches so far


[48/48] stg_ss7_20260803_2300.csv: 261057 rows, 53615 in scope, 48 files with matches so far

total rows scanned: 14,685,436
rows in scope (message_type in [2, 3]): 3,417,425
in-scope rows with a real payload but no decoded_content (not searched): 40
total pattern matches (row-level, token anchored to end of message): 273,444


## Summary tables

In [3]:
print("total match count:", len(matches))

total match count: 273444


In [4]:
print("match count by message_type:")
matches["message_type"].value_counts().sort_index()

match count by message_type:


message_type
2        57
3    273387
Name: count, dtype: int64

In [5]:
print("match count by dcs:")
matches["dcs"].value_counts().sort_index()

match count by dcs:


dcs
0    136276
4    136228
8       940
Name: count, dtype: int64

In [6]:
calling_gt_counts = matches["calling_gt"].value_counts()
print(f"distinct callingGT: {calling_gt_counts.shape[0]}")
calling_gt_counts

distinct callingGT: 88


calling_gt
60120000062     62304
60120000061     61391
60120000060     60510
60120000665     30608
60120000664     30603
                ...  
60120000087         1
6281106007          1
601110499990        1
8615644274          1
60120000088         1
Name: count, Length: 88, dtype: int64

In [7]:
called_gt_counts = matches["called_gt"].value_counts()
print(f"distinct calledGT: {called_gt_counts.shape[0]}")
called_gt_counts

distinct calledGT: 12


called_gt
60120000015          273374
60120000062              23
60120000061              22
60120000060              12
601200000151              2
601200000155              2
60120000015111            2
6012000001551656          2
60120000015258            2
60120000015994853         1
60120000015585            1
601200000155254           1
Name: count, dtype: int64

In [8]:
gt_pair_counts = matches.groupby(["calling_gt", "called_gt"]).size().sort_values(ascending=False)
print(f"distinct (callingGT, calledGT) pairs: {gt_pair_counts.shape[0]}")
gt_pair_counts

distinct (callingGT, calledGT) pairs: 106


calling_gt     called_gt  
60120000062    60120000015    62299
60120000061    60120000015    61387
60120000060    60120000015    60510
60120000665    60120000015    30605
60120000664    60120000015    30603
                              ...  
601110499990   60120000060        1
628160610000   60120000015        1
66923012125    60120000015        1
923330066007   60120000015        1
9779851999115  60120000015        1
Length: 106, dtype: int64

In [9]:
print("dcs distribution split by message_type (matched rows only):")
pd.crosstab(matches["message_type"], matches["dcs"])

dcs distribution split by message_type (matched rows only):


dcs,0,4,8
message_type,,,
2,48,0,9
3,136228,136228,931


## Full match list

In [10]:
full_match_list = matches[["time_stamp", "message_type", "dcs", "calling_gt", "called_gt", "matched_token", "reference"]]
full_match_list

,time_stamp,message_type,dcs,calling_gt,called_gt,matched_token,reference
0,2026-08-02 00:01:49,3,4,60120000663,60120000015,(T2ZKaMjKhf4w),3091069221
1,2026-08-02 00:00:20,3,0,60120000062,60120000015,(0MWdcbeN8Qqj),2514995366
2,2026-08-02 00:00:24,3,4,60120000062,60120000015,(rzw50MprZHhH),3091205152
3,2026-08-02 00:00:24,3,4,60120000665,60120000015,(dZCRSF1Lmsdk),3091068901
4,2026-08-02 00:00:00,3,0,60120000062,60120000015,(X2FNV2oi72oC),2514121466
...,...,...,...,...,...,...,...
273439,2026-08-03 23:01:50,3,0,60120000061,60120000015,(MLtnbMgK6OlW),2524812188
273440,2026-08-03 23:01:48,3,4,60120000061,60120000015,(C15nfzsJp321),3098065806
273441,2026-08-03 23:01:48,3,4,60120000062,60120000015,(h93ByudQ15pH),2524814590
273442,2026-08-03 23:01:51,3,0,60120000664,60120000015,(yHsaXA0SVAgO),3102635284


In [11]:
out_path = PROJECT_ROOT / "notebooks" / "ss7_regex_content_matches.csv"
full_match_list.to_csv(out_path, index=False)
print(f"wrote {len(full_match_list)} rows to {out_path}")

wrote 273444 rows to C:\Users\IshitaGodani\Documents\projects\spam-detection-prototype\notebooks\ss7_regex_content_matches.csv
